# Week 2 — EDA & Feature Engineering
**AI-Powered WAF | CSIC 2010 HTTP Dataset**

This notebook covers:
1. Parsing raw HTTP request files
2. Extracting numeric features
3. Exploratory Data Analysis (EDA)
4. Saving the processed dataset to `data/processed.csv`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for nbconvert
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

print('Libraries loaded OK')

## 1. Parse Raw HTTP Files

In [ ]:
from src.data_parser import load_dataset

df_raw = load_dataset()

In [ ]:
df_raw[['method', 'path', 'query_string', 'body', 'content_length', 'has_cookie', 'label']].head(5)

In [ ]:
print('Shape:', df_raw.shape)
print('\nColumn types:')
print(df_raw.dtypes)
print('\nNull counts:')
print(df_raw.isnull().sum())

## 2. Class Distribution

In [ ]:
vc = df_raw['label'].value_counts()       # index: 0 and 1
n_normal = vc[0]
n_attack = vc[1]
print(f'Normal : {n_normal:,}')
print(f'Attack : {n_attack:,}')
print(f'Imbalance ratio: {n_normal/n_attack:.2f}x more normal than attack')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['Normal (0)', 'Attack (1)'], [n_normal, n_attack],
            color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Class Distribution (Count)')
axes[0].set_ylabel('Number of Requests')
for i, v in enumerate([n_normal, n_attack]):
    axes[0].text(i, v + 100, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie([n_normal, n_attack], labels=['Normal', 'Attack'], autopct='%1.1f%%',
            colors=['steelblue', 'tomato'], startangle=90)
axes[1].set_title('Class Distribution (%)')

plt.tight_layout()
plt.savefig('../data/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved class_distribution.png')

## 3. HTTP Method Distribution

In [ ]:
method_label = df_raw.groupby(['method', 'label']).size().unstack(fill_value=0)
method_label.columns = ['Normal', 'Attack']
print(method_label)

method_label.plot(kind='bar', color=['steelblue', 'tomato'], edgecolor='black', figsize=(8, 4))
plt.title('HTTP Method vs Class')
plt.xlabel('HTTP Method')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(title='Class')
plt.tight_layout()
plt.show()

## 4. Feature Extraction

In [ ]:
from src.feature_extractor import extract_features

df_feat = extract_features(df_raw)
df_feat = df_feat.reset_index(drop=True)
df_raw  = df_raw.reset_index(drop=True)

print('Feature matrix shape:', df_feat.shape)
df_feat.head()

In [ ]:
df_feat.describe().round(2)

## 5. Feature Distributions: Normal vs Attack

In [ ]:
numeric_feats = ['url_length', 'query_length', 'body_length',
                 'num_query_params', 'content_length', 'special_char_count']

normal  = df_feat[df_feat['label'] == 0]
attacks = df_feat[df_feat['label'] == 1]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(numeric_feats):
    axes[i].hist(normal[feat].clip(upper=normal[feat].quantile(0.99)),
                 bins=40, alpha=0.6, color='steelblue', label='Normal', density=True)
    axes[i].hist(attacks[feat].clip(upper=attacks[feat].quantile(0.99)),
                 bins=40, alpha=0.6, color='tomato', label='Attack', density=True)
    axes[i].set_title(feat)
    axes[i].legend()

plt.suptitle('Feature Distributions: Normal vs Attack', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved feature_distributions.png')

## 6. Attack Pattern Flags

In [ ]:
pattern_cols = ['has_sql', 'has_xss', 'has_path_traversal', 'has_cmd_injection', 'has_null_byte']

pattern_rates = pd.DataFrame({
    'Normal':  normal[pattern_cols].mean(),
    'Attack':  attacks[pattern_cols].mean()
}) * 100

print('Detection rate (%) per pattern:')
print(pattern_rates.round(2))

pattern_rates.plot(kind='bar', color=['steelblue', 'tomato'], edgecolor='black', figsize=(10, 4))
plt.title('Attack Pattern Detection Rate (%) by Class')
plt.ylabel('% of requests matching pattern')
plt.xlabel('Pattern')
plt.xticks(rotation=20)
plt.legend(title='Class')
plt.tight_layout()
plt.savefig('../data/pattern_detection_rates.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved pattern_detection_rates.png')

## 7. Correlation Heatmap

In [ ]:
corr = df_feat.corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.7}
)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved correlation_heatmap.png')

## 8. Top Correlated Features with Label

In [ ]:
label_corr = corr['label'].drop('label').sort_values(key=abs, ascending=False)
print('Feature correlation with label (attack=1):')
print(label_corr.round(3).to_string())

label_corr.plot(
    kind='barh',
    color=['tomato' if v > 0 else 'steelblue' for v in label_corr],
    edgecolor='black',
    figsize=(9, 6)
)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Feature Correlation with Label (Attack=1)', fontweight='bold')
plt.xlabel('Pearson Correlation')
plt.tight_layout()
plt.savefig('../data/label_correlations.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved label_correlations.png')

## 9. Sample Attack Requests

In [ ]:
# Show top 3 attack requests by special char count
attack_idx = df_feat[df_feat['label'] == 1]['special_char_count'].nlargest(3).index

for idx in attack_idx:
    row_raw  = df_raw.iloc[idx]
    row_feat = df_feat.iloc[idx]
    print('='*70)
    print(f"Method       : {row_raw['method']}")
    print(f"Path         : {row_raw['path']}")
    print(f"Query        : {str(row_raw['query_string'])[:120]}")
    print(f"Body         : {str(row_raw['body'])[:120]}")
    print(f"SpecialChars : {row_feat['special_char_count']}")

## 10. Save Processed Dataset

In [ ]:
output_path = '../data/processed.csv'
df_feat.to_csv(output_path, index=False)
print(f'Saved {len(df_feat):,} rows x {len(df_feat.columns)} columns -> {output_path}')
print('\nFeatures saved:')
for col in df_feat.columns:
    print(f'  {col}')

## Summary

| Item | Value |
|------|-------|
| Total requests | 61,065 |
| Normal | 36,000 |
| Attack | 25,065 |
| Features extracted | 16 numeric |
| Output file | `data/processed.csv` |

**Next (Week 3):** Load `processed.csv`, train a Random Forest classifier, evaluate with precision/recall/F1, and tune the decision threshold.